# Exercise P3.1: API Authentication & Pagination
### STAT 540 — Week 3


## Overview

In this exercise, you will query the GitHub API with pagination to collect a larger dataset, handle rate limits, and store the results in a clean DataFrame.

## Task 1: Basic API Query

In [1]:
import requests
import pandas as pd

# Query top-starred repos
response = requests.get(
    "https://api.github.com/search/repositories",
    params={"q": "stars:>10000", "sort": "stars", "per_page": 30, "page": 1}
)

print(f"Status: {response.status_code}")
data = response.json()
print(f"Total count: {data['total_count']}")
print(f"Items returned: {len(data['items'])}")

Status: 200
Total count: 5542
Items returned: 30


## Task 2: Paginate to Collect 100 Repos

In [2]:
import requests
import pandas as pd
import time

all_repos = []
for page in range(1, 4):   # 3 pages × 30 = up to 90 repos
    response = requests.get(
        "https://api.github.com/search/repositories",
        params={"q": "stars:>10000", "sort": "stars", "per_page": 30, "page": page}
    )
    if response.status_code != 200:
        print(f"Error on page {page}: {response.status_code}")
        break

    items = response.json().get("items", [])
    all_repos.extend(items)
    print(f"Page {page}: collected {len(items)}, total: {len(all_repos)}")
    time.sleep(1)   # Polite delay

df = pd.json_normalize(all_repos)
print(f"\nFinal shape: {df.shape}")

Page 1: collected 30, total: 30
Page 2: collected 30, total: 60
Page 3: collected 30, total: 90

Final shape: (90, 105)


## Task 3: Clean and Analyze

In [5]:
# Select useful columns
df_clean = df[[
    "full_name", "stargazers_count", "forks_count",
    "language", "description", "created_at", "owner.login"
]].copy()

df_clean.columns = ["repo", "stars", "forks", "language", "description", "created", "owner"]
df_clean["created"] = pd.to_datetime(df_clean["created"])

# Top 10 by stars
print(df_clean.nlargest(10, "stars")[["repo", "stars", "language"]])

# Most common languages
print(df_clean["language"].value_counts().head(10))

# Most common year
df_clean["year_created"] = df_clean["created"].dt.year
print(df_clean["year_created"].value_counts().head(1))

                                     repo   stars    language
0        codecrafters-io/build-your-own-x  546064    Markdown
1                    sindresorhus/awesome  504276        None
2                 public-apis/public-apis  477615      Python
3               freeCodeCamp/freeCodeCamp  455236  TypeScript
4  EbookFoundation/free-programming-books  396301      Python
5                       openclaw/openclaw  389246  TypeScript
6        donnemartin/system-design-primer  368816      Python
7              nilbuild/developer-roadmap  366629  TypeScript
8     jwasham/coding-interview-university  360600        None
9                    vinta/awesome-python  319384      Python
language
Python        24
TypeScript    16
JavaScript     9
Shell          4
Go             4
Rust           4
C++            4
HTML           3
C              3
Markdown       3
Name: count, dtype: int64
year_created
2025    12
Name: count, dtype: int64


**Your turn:** What is the most popular language among top-starred repos? What year were most of these repos created?

> The most popular language among the top-starred repos is Python, and most of these repos were created in 2025.

## Task 4: Check Rate Limits

In [7]:
import requests

rate = requests.get("https://api.github.com/rate_limit").json()["rate"]
print(f"Limit: {rate['limit']}")
print(f"Remaining: {rate['remaining']}")

Limit: 60
Remaining: 60


**Your turn:** How many requests do you have remaining? What happens when you hit 0?

> I have 60 requests remaining. When I hit 0, the API will return an error status (normally 403) and will not serve the request, so I will have to wait until the limit rests at a certain time.

## Task 5: Try a Different API

Choose a different public API, query it, and convert to a DataFrame:

- **Open Trivia DB:** `https://opentdb.com/api.php?amount=20&category=18`
- **Random User:** `https://randomuser.me/api/?results=20`
- **REST Countries:** `https://restcountries.com/v3.1/all`

In [8]:
import requests
import pandas as pd

# Choose the Random User API
api_url = "https://randomuser.me/api/?results=20"

# Make the API request
response = requests.get(api_url)

# Check for successful response
if response.status_code == 200:
    data = response.json()
    # Normalize the JSON data into a DataFrame
    df_random_users = pd.json_normalize(data['results'])
    print(f"Collected {len(data['results'])} random users.")
    display(df_random_users.head())
else:
    print(f"Error fetching data: {response.status_code}")

Collected 20 random users.


,gender,email,phone,cell,nat,name.title,name.first,name.last,location.street.number,location.street.name,...,login.sha256,dob.date,dob.age,registered.date,registered.age,id.name,id.value,picture.large,picture.medium,picture.thumbnail
0,female,evelyn.kumar@example.com,(595)-690-5443,(275)-286-8373,NZ,Miss,Evelyn,Kumar,9386,Hanover Street,...,b369ebecfb5fb6f8ecb8dab100ba6aa865a75b8a47f313...,1995-05-15T16:04:48.425Z,31,2006-02-11T10:51:32.131Z,20,,None,https://randomuser.me/api/portraits/women/96.jpg,https://randomuser.me/api/portraits/med/women/...,https://randomuser.me/api/portraits/thumb/wome...
1,male,sylvain.fleury@example.com,078 245 83 78,075 702 04 59,CH,Monsieur,Sylvain,Fleury,831,Rue Paul Bert,...,e97ade5473687ef757ac35adff9875c1ec7759b0080431...,1966-02-23T08:29:19.592Z,60,2020-01-30T15:56:10.450Z,6,AVS,756.3908.9069.35,https://randomuser.me/api/portraits/men/19.jpg,https://randomuser.me/api/portraits/med/men/19...,https://randomuser.me/api/portraits/thumb/men/...
2,female,djuna.vanlangeveld@example.com,(0978) 760550,(06) 37357467,NL,Mrs,Djuna,Van Langeveld,8321,Holtgesbroek,...,16d19fe1408e261b52b60d7383cb787040eb78b1510c0c...,1986-02-26T03:43:04.659Z,40,2019-01-01T19:25:11.523Z,7,BSN,21382106,https://randomuser.me/api/portraits/women/7.jpg,https://randomuser.me/api/portraits/med/women/...,https://randomuser.me/api/portraits/thumb/wome...
3,female,lisa.griffin@example.com,041-451-8911,081-555-3383,IE,Miss,Lisa,Griffin,8359,The Green,...,6b994f8825bd66eb8c7391fb980b622263a3c177b55fcc...,1994-09-12T00:07:34.361Z,31,2020-12-21T01:40:57.568Z,5,PPS,2446433T,https://randomuser.me/api/portraits/women/86.jpg,https://randomuser.me/api/portraits/med/women/...,https://randomuser.me/api/portraits/thumb/wome...
4,male,rajko.peric@example.com,028-6369-627,066-7148-816,RS,Mr,Rajko,Perić,2193,Stevana Stevanovića,...,eaa9e826d4c92abb4b4b4768090211e91f01e9cf958345...,1976-04-29T09:10:26.986Z,50,2017-03-06T06:49:04.215Z,9,SID,878087153,https://randomuser.me/api/portraits/men/66.jpg,https://randomuser.me/api/portraits/med/men/66...,https://randomuser.me/api/portraits/thumb/men/...


**Your turn:** What API did you choose? Describe the data you received.

> I chose the Random User API. The data received is a DataFrame named df_random_users containing information for 20 randomly generated users. Each row represents a unique user, and there are 34 columns detailing various attributes such as personal details, location information, login details, and ID/pictures.

## Submission

```bash
git add week03/exercises/P3.1*
git commit -m "Complete Exercise P3.1: API pagination with Python"
git push origin main
```